In [ ]:
# Inference throughput on Kaggle's T4 x2 machine (same image as knee-submit), on N training studies posing as a test set.
# Question: does knee-submit's 8 h budget cut our ensembles on the hidden test, and do the CPUs or the GPU limit it?
import glob, os, subprocess, sys, time, threading
T0 = time.time()
def find(pattern):
    return next(p for depth in range(4) for p in glob.glob('/kaggle/input/' + '*/' * depth + pattern))
KNEE = os.path.dirname(os.path.dirname(find('knee/infer.py')))
DATA = os.path.dirname(find('train_series.csv'))
print(KNEE, DATA)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', f'{KNEE}/wheels',
                'pylibjpeg', 'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg', 'python-gdcm'], check=False)
sys.path.insert(0, KNEE)
import numpy as np, pandas as pd, torch
from pathlib import Path
print('GPUs', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'CPUs', os.cpu_count(),
      'sched CPUs', len(os.sched_getaffinity(0)))
subprocess.run(['nproc'])
subprocess.run(['free', '-g'])

In [ ]:
# Version 2: the repository's current knee/infer.py, data.py, model.py (grouped two-GPU inference) replace the
# dataset's copies, then the whole Version 27 list runs grouped on 60 training studies posing as a test set.
import shutil
KN = '/kaggle/working/kn'
shutil.rmtree(KN, ignore_errors=True)
shutil.copytree(f'{KNEE}/knee', f'{KN}/knee')
NEW = {"knee/infer.py": "\"\"\"Test-time pipeline (Kaggle notebook, internet off): DICOM folders -> submission.csv.\n\nPreprocessing is the training code itself (knee/dicom_io.py + knee/cache_io.py), so train and\ntest inputs are identical. Every series is decoded once into a local cache (one per slice size\nthe models were trained on, see config \"cache_size\"), then each model reads its cache.\n\n    from knee.infer import run\n    run(data_dir=\"/kaggle/input/competitions/rsna-knee-abnormality-detection\",\n        model_dirs=[\"/kaggle/input/knee-models/models/r34_base\"], out_csv=\"submission.csv\")\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport time\nfrom concurrent.futures import ProcessPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pydicom\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom knee.cache_io import SUFFIX, save_series\nfrom knee.config import LABELS\nfrom knee.data import StudyDataset, assign_slots\nfrom knee.dicom_io import load_volume, series_side_info, study_side, to_uint8\nfrom knee.model import KneeNet\n\n\ndef _side_info(series_dir):\n    try:\n        headers = [pydicom.dcmread(p, stop_before_pixels=True, force=True) for p in Path(series_dir).glob(\"*.dcm\")]\n        return series_side_info(headers)\n    except Exception:\n        return {\"side\": \"U\", \"source\": \"none\", \"center_x\": None}\n\n\ndef size_dir(cache_dir: Path, size: int) -> Path:\n    \"\"\"Test cache of one slice size; the 256 px one keeps the plain path.\"\"\"\n    return cache_dir if size == 256 else cache_dir.with_name(f\"{cache_dir.name}{size}\")\n\n\ndef _prepare(args):\n    series_dir, out_dir, study, series, side, sizes = args\n    try:\n        blobs = [p.read_bytes() for p in sorted(Path(series_dir).glob(\"*.dcm\"))]\n        vol = load_volume(blobs, side=side)\n        for size in sizes:  # as scripts/build_cache.py --size\n            x, pos = to_uint8(vol, size=size)\n            (size_dir(out_dir, size) / study).mkdir(parents=True, exist_ok=True)\n            save_series(size_dir(out_dir, size) / study / f\"{series}{SUFFIX}\", x, pos)\n        return {\"study\": study, \"series\": series, \"cached_slices\": len(x), \"ok\": True,  # + header fields of series_meta\n                **{t: vol.meta.get(t) for t in (\"MRAcquisitionType\", \"RepetitionTime\", \"EchoTime\", \"InversionTime\",\n                                                \"ScanningSequence\")}}\n    except Exception as e:  # a broken series must not sink the submission\n        return {\"study\": study, \"series\": series, \"ok\": False, \"error\": repr(e)}\n\n\ndef prepare_cache(data_dir: Path, cache_dir: Path, workers: int, sizes=(256,)) -> pd.DataFrame:\n    series = pd.read_csv(data_dir / \"test_series.csv\")\n    dirs = [data_dir / \"test_series\" / st / se for st, se in zip(series[\"StudyInstanceUID\"], series[\"SeriesInstanceUID\"])]\n    t = time.time()\n    with ProcessPoolExecutor(workers) as pool:\n        # headers first: one knee side per study, exactly as the training cache (build_cache --fix-laterality)\n        infos = list(pool.map(_side_info, dirs, chunksize=4))\n        by_study: dict = {}\n        for st, info in zip(series[\"StudyInstanceUID\"], infos):\n            by_study.setdefault(st, []).append(info)\n        sides = {st: study_side(v) for st, v in by_study.items()}\n        print(f\"headers read in {time.time() - t:.0f}s, sides {pd.Series(sides).value_counts().to_dict()}\", flush=True)\n        jobs = [(d, cache_dir, st, se, sides[st], sizes)\n                for d, st, se in zip(dirs, series[\"StudyInstanceUID\"], series[\"SeriesInstanceUID\"])]\n        rows = list(pool.map(_prepare, jobs, chunksize=4))\n    meta = pd.DataFrame(rows)\n    print(f\"decoded {int(meta['ok'].sum())}/{len(meta)} series in {time.time() - t:.0f}s\", flush=True)\n    if (~meta[\"ok\"]).any():\n        print(meta.loc[~meta[\"ok\"], \"error\"].value_counts().head(), flush=True)\n    df = series.merge(meta[meta[\"ok\"]], left_on=[\"StudyInstanceUID\", \"SeriesInstanceUID\"], right_on=[\"study\", \"series\"])\n    df[\"csv_plane\"] = df[\"Anatomical_Plane\"].str.lower()\n    return df\n\n\n@torch.no_grad()\ndef predict_model(model_dir: Path, slots: pd.DataFrame, cache_dir: Path, device, workers: int,\n                  windows: int | None = None) -> np.ndarray:\n    cfg = json.loads((model_dir / \"config.json\").read_text())\n    windows = windows or cfg[\"windows_eval\"]  # denser than in training helps a little (experiments.md, E17)\n    ds = StudyDataset(slots, n_windows=windows, img_size=cfg[\"img_size\"],\n                      span=tuple(cfg.get(\"span\", (0.08, 0.92))), zoom=cfg.get(\"zoom\", 1.0), cache_dir=cache_dir)\n    dl = DataLoader(ds, batch_size=max(1, 48 // windows), num_workers=workers)\n    models = []\n    for ckpt in sorted(model_dir.glob(\"fold*.pt\")):\n        model = KneeNet(cfg[\"backbone\"], pretrained=False, dim=cfg[\"dim\"], depth=cfg[\"depth\"], drop_path=0,\n                         pool=cfg.get(\"pool\", \"avg\"), seq_emb=cfg.get(\"seq_emb\", False))\n        model.load_state_dict(torch.load(ckpt, map_location=\"cpu\"))\n        models.append(model.to(device).eval().to(memory_format=torch.channels_last))\n    out = [[] for _ in models]\n    for batch in dl:  # every batch is read once and goes through all fold networks\n        x, pos, mask, seq = (batch[k].to(device) for k in (\"x\", \"pos\", \"mask\", \"seq\"))\n        for i, model in enumerate(models):\n            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == \"cuda\"):\n                logits = model(x, pos, mask, seq)\n            out[i].append(torch.sigmoid(logits.float()).cpu().numpy())\n    print(f\"{model_dir.name}: {len(models)} fold networks done\", flush=True)\n    return np.mean([np.concatenate(o) for o in out], axis=0)\n\n\n# forward cost per image relative to ConvNeXt-tiny at the same slice size (FLOPs), for the time budget\nBACKBONE_COST = {\"convnext_small\": 1.95, \"radimagenet_resnet50\": 0.92, \"resnet50\": 0.92, \"resnet34\": 0.8}\n\n\ndef model_cost(model_dir: Path, cfg: dict, windows: int | None = None) -> float:\n    \"\"\"Relative inference cost of one model per study: pixels x windows x fold networks x backbone factor.\"\"\"\n    factor = next((v for k, v in BACKBONE_COST.items() if cfg[\"backbone\"].startswith(k)), 1.0)\n    return cfg[\"img_size\"] ** 2 * (windows or cfg[\"windows_eval\"]) * len(list(model_dir.glob(\"fold*.pt\"))) * factor\n\n\ndef input_key(cfg: dict, windows: int | None = None) -> tuple:\n    \"\"\"Models with the same key read identical batches: cache, slice size, windows, span, zoom.\"\"\"\n    return (cfg.get(\"cache_size\", 256), cfg[\"img_size\"], windows or cfg[\"windows_eval\"],\n            tuple(cfg.get(\"span\", (0.08, 0.92))), cfg.get(\"zoom\", 1.0))\n\n\n@torch.no_grad()\ndef predict_group(model_dirs: list, cfgs: dict, slots: pd.DataFrame, cache_dir: Path, devices: list, workers: int,\n                  windows: int | None = None) -> tuple[dict, float, float]:\n    \"\"\"predict_model for several models that read the same input (input_key): each batch is read once and\n    goes through every fold network of every model, the networks spread round-robin over the GPUs, so\n    consecutive launches alternate between them and both compute at once.\n    Returns ({model_dir: mean fold probability}, seconds loading networks, seconds predicting).\"\"\"\n    cfg = cfgs[model_dirs[0]]\n    windows = windows or cfg[\"windows_eval\"]\n    ds = StudyDataset(slots, n_windows=windows, img_size=cfg[\"img_size\"],\n                      span=tuple(cfg.get(\"span\", (0.08, 0.92))), zoom=cfg.get(\"zoom\", 1.0), cache_dir=cache_dir)\n    dl = DataLoader(ds, batch_size=max(1, 48 // windows), num_workers=workers, pin_memory=devices[0].type == \"cuda\")\n    t = time.time()\n    nets = []  # (model_dir, device, network)\n    for m in model_dirs:\n        c = cfgs[m]\n        for ckpt in sorted(m.glob(\"fold*.pt\")):\n            dev = devices[len(nets) % len(devices)]\n            net = KneeNet(c[\"backbone\"], pretrained=False, dim=c[\"dim\"], depth=c[\"depth\"], drop_path=0,\n                          pool=c.get(\"pool\", \"avg\"), seq_emb=c.get(\"seq_emb\", False))\n            net.load_state_dict(torch.load(ckpt, map_location=\"cpu\"))\n            nets.append((m, dev, net.to(dev).eval().to(memory_format=torch.channels_last)))\n    load_s, t = time.time() - t, time.time()\n    out = {m: [] for m in model_dirs}\n    for batch in dl:\n        inputs = {d: [batch[k].to(d, non_blocking=True) for k in (\"x\", \"pos\", \"mask\", \"seq\")] for d in devices}\n        logits = []\n        for m, d, net in nets:  # launches only; .cpu() below waits for the GPUs\n            with torch.autocast(d.type, dtype=torch.float16, enabled=d.type == \"cuda\"):\n                logits.append((m, net(*inputs[d])))\n        folds = {m: [] for m in model_dirs}\n        for m, lg in logits:\n            folds[m].append(torch.sigmoid(lg.float()).cpu().numpy())\n        for m in model_dirs:\n            out[m].append(np.mean(folds[m], axis=0))\n    for m in model_dirs:\n        print(f\"{m.name}: {sum(n[0] == m for n in nets)} fold networks done\", flush=True)\n    return {m: np.concatenate(out[m]) for m in model_dirs}, load_s, time.time() - t\n\n\ndef run(data_dir, model_dirs, out_csv=\"submission.csv\", cache_dir=\"/tmp/knee_cache\", workers=None,\n        time_budget_s=None, windows=None, grouped=False, calib_studies=16):\n    \"\"\"time_budget_s: skip the remaining models (in the given order) once the next one would end past\n    this many seconds from the call; the blend then uses the models done so far.\n    grouped: models sharing an input run together on every GPU (predict_group); a first pass over\n    calib_studies studies measures the speed, the models that fit the budget are chosen in the given\n    order, and before each group the slowest-priority ones are dropped if the measured pace says so.\"\"\"\n    t0 = time.time()\n    data_dir, cache_dir = Path(data_dir), Path(cache_dir)\n    workers = workers or os.cpu_count()\n    test = pd.read_csv(data_dir / \"test.csv\")\n    model_dirs = [Path(m) for m in model_dirs]\n    cfgs = {m: json.loads((m / \"config.json\").read_text()) for m in model_dirs}\n    sizes = {m: c.get(\"cache_size\", 256) for m, c in cfgs.items()}\n    series = prepare_cache(data_dir, cache_dir, workers, sorted(set(sizes.values())))\n    slots = assign_slots(series).set_index(\"StudyInstanceUID\").reindex(test[\"StudyInstanceUID\"]).reset_index()\n    if grouped:\n        preds = _run_grouped(model_dirs, cfgs, slots, cache_dir, workers, windows, time_budget_s, t0, calib_studies)\n        return _write(test, [preds[m] for m in model_dirs if m in preds], len(model_dirs), out_csv)\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    preds, spent, pixels = [], 0.0, 0\n    for model_dir in map(Path, model_dirs):\n        cost = cfgs[model_dir][\"img_size\"] ** 2 * len(list(model_dir.glob(\"fold*.pt\"))) * (windows or cfgs[model_dir][\"windows_eval\"])\n        if preds and time_budget_s and time.time() - t0 + spent / pixels * cost * 1.2 > time_budget_s:\n            print(f\"time budget: skipping {model_dir.name}\", flush=True)\n            continue\n        t = time.time()\n        p = predict_model(model_dir, slots, size_dir(cache_dir, sizes[model_dir]), device, workers, windows)\n        spent, pixels = spent + time.time() - t, pixels + cost\n        print(f\"{model_dir.name}: {time.time() - t:.0f}s, {(time.time() - t0) / 60:.1f} min elapsed\", flush=True)\n        preds.append(p)\n    return _write(test, preds, len(model_dirs), out_csv)\n\n\ndef _write(test: pd.DataFrame, preds: list, n_models: int, out_csv) -> pd.DataFrame:\n    # several models: average ranks, so no model dominates through its calibration\n    preds = [pd.DataFrame(p).rank(pct=True).to_numpy() for p in preds] if n_models > 1 else preds\n    sub = pd.DataFrame(np.mean(preds, axis=0), columns=LABELS)\n    sub.insert(0, \"StudyInstanceUID\", test[\"StudyInstanceUID\"].to_numpy())\n    sub[LABELS] = sub[LABELS].fillna(0.5)\n    sub.to_csv(out_csv, index=False)\n    print(sub.head())\n    return sub\n\n\ndef _run_grouped(model_dirs, cfgs, slots, cache_dir, workers, windows, time_budget_s, t0, calib_studies) -> dict:\n    devices = [torch.device(f\"cuda:{i}\") for i in range(torch.cuda.device_count())] or [torch.device(\"cpu\")]\n    print(f\"devices: {[str(d) for d in devices]}\", flush=True)\n    costs = {m: model_cost(m, cfgs[m], windows) for m in model_dirs}\n    groups: dict = {}  # input key -> its models, in the given order; groups run in the order of their first model\n    for m in model_dirs:\n        groups.setdefault(input_key(cfgs[m], windows), []).append(m)\n    n = len(slots)\n    keep = list(model_dirs)\n    rate, load = None, 0.0  # cost units x studies per second (all GPUs together); seconds to load one network\n    if time_budget_s and n > calib_studies:  # speed of the first group on a few studies\n        key, ms = next(iter(groups.items()))\n        _, load_s, pred_s = predict_group(ms, cfgs, slots.iloc[:calib_studies], size_dir(cache_dir, key[0]), devices,\n                                          workers, windows)\n        rate = sum(costs[m] for m in ms) * calib_studies / pred_s\n        load = load_s / sum(len(list(m.glob(\"fold*.pt\"))) for m in ms)\n        left = time_budget_s - (time.time() - t0)\n        keep, total = [], 0.0\n        for m in model_dirs:  # given order = priority\n            need = (costs[m] * n / rate + load * len(list(m.glob(\"fold*.pt\")))) * 1.15\n            if not keep or total + need <= left:\n                keep.append(m)\n                total += need\n            else:\n                print(f\"time budget: skipping {m.name}\", flush=True)\n        print(f\"calibration on {calib_studies} studies: {rate:.3g} units/s, {load:.1f} s per network load; \"\n              f\"{total / 60:.0f} min planned for {len(keep)} models, {left / 60:.0f} min left\", flush=True)\n    preds, done, spent = {}, 0.0, 0.0\n    for key, ms in groups.items():\n        ms = [m for m in ms if m in keep]\n        if preds and time_budget_s and rate:  # measured pace: drop this group's last models if they no longer fit\n            left = time_budget_s - (time.time() - t0)\n            while ms and sum(costs[m] for m in ms) * n / rate * 1.15 > left:\n                print(f\"time budget: skipping {ms.pop().name}\", flush=True)\n        if not ms:\n            continue\n        t = time.time()\n        p, _, pred_s = predict_group(ms, cfgs, slots, size_dir(cache_dir, key[0]), devices, workers, windows)\n        preds.update(p)\n        done, spent = done + sum(costs[m] for m in ms) * n, spent + pred_s\n        rate = done / spent\n        print(f\"group {key[:3]}: {len(ms)} models in {time.time() - t:.0f}s, {(time.time() - t0) / 60:.1f} min elapsed\",\n              flush=True)\n    return preds\n", "knee/data.py": "\"\"\"Study-level dataset: a few series (\"slots\") per study, several 2.5D windows per series.\n\nA window is 3 adjacent slices stacked as RGB, so an ImageNet 2D backbone sees local 3D context.\nFrames are already canonical (knee/dicom_io.py): no horizontal flip here, it would swap\nmedial and lateral.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom knee.cache_io import SUFFIX, RawSeriesReader, SeriesReader\nfrom knee.config import CACHE_DIR, DATA_DIR\n\n# (name, plane, fluid-sensitive fat-suppressed?)\nSLOTS = [(\"sag_fs\", \"sagittal\", 1), (\"cor_fs\", \"coronal\", 1), (\"ax_fs\", \"axial\", 1),\n         (\"sag_t\", \"sagittal\", 0), (\"cor_t\", \"coronal\", 0)]\n# weighting of the series in a slot, from its header times (KneeNet --seq-emb): the sag_t slot alone\n# holds T1 (39 %), PD (41 %) and T2 (20 %) series of the training set\nSEQ_CLASSES = [\"unknown\", \"T1\", \"PD\", \"T2\", \"STIR\", \"GRE\"]\n\n\ndef series_table(split: str = \"train\") -> pd.DataFrame:\n    \"\"\"Cached series with their CSV descriptors (plane, fat suppression) and geometry meta.\"\"\"\n    meta = pd.read_parquet(DATA_DIR / \"series_meta.parquet\")\n    csv = pd.read_csv(DATA_DIR / f\"{split}_series.csv\")\n    df = csv.merge(meta, left_on=[\"StudyInstanceUID\", \"SeriesInstanceUID\"], right_on=[\"study\", \"series\"], how=\"inner\")\n    df[\"csv_plane\"] = df[\"Anatomical_Plane\"].str.lower()\n    return df\n\n\ndef sequence_classes(series: pd.DataFrame) -> pd.Series:\n    \"\"\"Index into SEQ_CLASSES per series: TI 80-300 ms -> STIR, gradient echo -> GRE, TR < 1000 ms -> T1,\n    TE >= 60 ms -> T2, else PD; unknown when TR or TE is missing.\"\"\"\n    def num(col):\n        return pd.to_numeric(series[col], errors=\"coerce\") if col in series else pd.Series(float(\"nan\"), index=series.index)\n    tr, te, ti = num(\"RepetitionTime\"), num(\"EchoTime\"), num(\"InversionTime\")\n    gre = series[\"ScanningSequence\"].astype(str).eq(\"GR\") if \"ScanningSequence\" in series else False\n    cls = np.select([tr.isna() | te.isna(), ti.between(80, 300), gre, tr < 1000, te >= 60], [0, 4, 5, 1, 3], 2)\n    return pd.Series(cls, index=series[\"SeriesInstanceUID\"].to_numpy())\n\n\ndef assign_slots(series: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"One series per slot per study (None when the study has no candidate), and its SEQ_CLASSES\n    index in \"<slot>_seq\".\n\n    Candidates match the CSV plane and fat-suppression flag; ties go to 2D acquisitions,\n    then to the series with the most slices. An empty slot borrows another unused series\n    of the same plane, since a second view of that plane beats nothing.\"\"\"\n    rows = []\n    for study, g in series.groupby(\"StudyInstanceUID\"):\n        g = g.assign(is2d=(g[\"MRAcquisitionType\"].fillna(\"2D\") == \"2D\").astype(int))\n        g = g.sort_values([\"is2d\", \"cached_slices\", \"SeriesInstanceUID\"], ascending=[False, False, True])\n        row, used = {\"StudyInstanceUID\": study}, set()\n        for name, plane, fs in SLOTS:\n            cand = g[(g[\"csv_plane\"] == plane) & (g[\"Fat_Suppression\"] == fs) & ~g[\"SeriesInstanceUID\"].isin(used)]\n            row[name] = cand[\"SeriesInstanceUID\"].iloc[0] if len(cand) else None\n            if row[name]:\n                used.add(row[name])\n        for name, plane, _ in SLOTS:\n            if row[name] is None:\n                cand = g[(g[\"csv_plane\"] == plane) & ~g[\"SeriesInstanceUID\"].isin(used)]\n                if len(cand):\n                    row[name] = cand[\"SeriesInstanceUID\"].iloc[0]\n                    used.add(row[name])\n        rows.append(row)\n    out, seq = pd.DataFrame(rows), sequence_classes(series)\n    for name, _, _ in SLOTS:\n        out[f\"{name}_seq\"] = out[name].map(seq).fillna(0).astype(int)\n    return out\n\n\ndef window_centers(n_slices: int, n_windows: int, lo: float, hi: float, jitter: float, rng) -> np.ndarray:\n    \"\"\"Evenly spaced slice indices over [lo, hi] of the stack, optionally jittered.\"\"\"\n    q = np.linspace(lo, hi, n_windows)\n    if jitter:\n        q = np.clip(q + rng.uniform(-jitter, jitter, n_windows), 0, 1)\n    return np.clip(np.round(q * (n_slices - 1)).astype(int), 0, n_slices - 1)\n\n\ndef _affine(rng, size_in: int, size_out: int, strength: float, zoom: float = 1.0) -> np.ndarray:\n    \"\"\"Input -> output pixel map: resize to size_out (zoom > 1 keeps only the central 1/zoom of the\n    cached field of view), plus a random rotation/zoom/shift in training.\"\"\"\n    angle = rng.uniform(-10, 10) * strength\n    scale = math.exp(rng.uniform(-0.1, 0.12) * strength) * size_out / size_in * zoom\n    tx, ty = rng.uniform(-0.05, 0.05, 2) * size_out * strength\n    m = cv2.getRotationMatrix2D((size_in / 2, size_in / 2), angle, scale)\n    m[:, 2] += (size_out - size_in) / 2 + tx, (size_out - size_in) / 2 + ty\n    return m\n\n\ndef _is_missing(series) -> bool:\n    return series is None or (isinstance(series, float) and np.isnan(series))\n\n\nclass StudyDataset(Dataset):\n    def __init__(self, slots: pd.DataFrame, labels: np.ndarray | None = None, weights: np.ndarray | None = None,\n                 n_windows: int = 10, img_size: int = 224, train: bool = False, aug_strength: float = 1.0,\n                 span: tuple[float, float] = (0.08, 0.92), zoom: float = 1.0, seed: int = 0, cache_dir=CACHE_DIR,\n                 raw_dir=None):\n        self.slots = slots.reset_index(drop=True)\n        self.labels, self.weights = labels, weights\n        self.n_windows, self.img_size, self.train = n_windows, img_size, train\n        self.aug_strength, self.span, self.zoom, self.seed = aug_strength, span, zoom, seed\n        self.cache_dir, self.raw_dir = cache_dir, raw_dir\n        self.epoch = 0\n\n    def __len__(self):\n        return len(self.slots)\n\n    def _load(self, study: str, series: str):\n        if self.raw_dir is not None:  # unpacked copy on local disk (scripts/unpack_cache.py): no JPEG decoding\n            return RawSeriesReader(self.raw_dir / study / f\"{series}.npy\")\n        return SeriesReader(self.cache_dir / study / f\"{series}{SUFFIX}\")\n\n    def __getitem__(self, i):\n        rng = np.random.default_rng((self.seed, self.epoch, i)) if self.train else np.random.default_rng(i)\n        row = self.slots.iloc[i]\n        study = row[\"StudyInstanceUID\"]\n        n_slots, t, s = len(SLOTS), self.n_windows, self.img_size\n        x = np.zeros((n_slots, t, 3, s, s), np.uint8)\n        pos = np.zeros((n_slots, t), np.float32)\n        mask = np.zeros(n_slots, bool)\n        seq = np.zeros(n_slots, np.int64)\n        for k, (name, _, _) in enumerate(SLOTS):\n            series = row[name]\n            if _is_missing(series):\n                continue\n            seq[k] = 0 if pd.isna(row.get(f\"{name}_seq\")) else int(row[f\"{name}_seq\"])\n            vol = self._load(study, series)\n            n = len(vol)\n            centers = window_centers(n, t, *self.span, jitter=0.03 if self.train else 0.0, rng=rng)\n            needed = np.unique(np.clip(np.concatenate([centers - 1, centers, centers + 1]), 0, n - 1))\n            frames, m = {}, None\n            for z in needed:  # each slice decoded and transformed once, even when windows share it\n                img = vol.slice(z)\n                if m is None:\n                    m = _affine(rng, img.shape[-1], s, self.aug_strength if self.train else 0.0, self.zoom)\n                frames[z] = cv2.warpAffine(img, m, (s, s), flags=cv2.INTER_LINEAR, borderValue=0)\n            for j, c in enumerate(centers):\n                for ch, z in enumerate(np.clip([c - 1, c, c + 1], 0, n - 1)):\n                    x[k, j, ch] = frames[z]\n            if self.train:  # gamma + contrast through a 256-entry lookup table\n                gamma = math.exp(rng.uniform(-0.25, 0.25) * self.aug_strength)\n                contrast = 1 + rng.uniform(-0.15, 0.15) * self.aug_strength\n                lut = np.clip(((np.arange(256) / 255.0) ** gamma - 0.5) * contrast + 0.5, 0, 1)\n                x[k] = (lut * 255).astype(np.uint8)[x[k]]\n            pos[k] = centers / max(n - 1, 1)\n            mask[k] = True\n        out = {\"x\": torch.from_numpy(x), \"pos\": torch.from_numpy(pos), \"mask\": torch.from_numpy(mask),\n               \"seq\": torch.from_numpy(seq)}\n        if self.labels is not None:\n            out[\"y\"] = torch.from_numpy(self.labels[i].astype(np.float32))\n            out[\"w\"] = torch.from_numpy(self.weights[i].astype(np.float32))\n        return out\n", "knee/model.py": "\"\"\"2.5D multi-series model: a shared 2D backbone embeds every window, a small transformer mixes\nthe windows of all series, and one learned query per finding pools the tokens it needs\n(e.g. sagittal windows for the ACL, coronal ones for the MCL).\"\"\"\n\nimport timm\nimport torch\nimport torch.nn as nn\n\nfrom knee.config import LABELS\nfrom knee.data import SEQ_CLASSES, SLOTS\n\nMEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1) * 255\nSTD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1) * 255\n\n\n# ResNet-50 pretrained on RadImageNet (1.35 M CT/MR/ultrasound images), MIT licence, pinned revision\nRADIMAGENET = (\"Lab-Rasool/RadImageNet\", \"ResNet50.pt\", \"14460ee4c1276f6925611a63aa9a54a05d39eae0\")\n\n\ndef make_backbone(name: str, pretrained: bool, **kwargs) -> nn.Module:\n    if name != \"radimagenet_resnet50\":\n        return timm.create_model(name, pretrained=pretrained, num_classes=0, **kwargs)\n    model = timm.create_model(\"resnet50\", pretrained=False, num_classes=0, **kwargs)\n    if pretrained:  # its keys are backbone.<i>.* of an nn.Sequential over the torchvision ResNet-50 children\n        from huggingface_hub import hf_hub_download\n        sd = torch.load(hf_hub_download(RADIMAGENET[0], RADIMAGENET[1], revision=RADIMAGENET[2]), map_location=\"cpu\")\n        names = {\"0\": \"conv1\", \"1\": \"bn1\", \"4\": \"layer1\", \"5\": \"layer2\", \"6\": \"layer3\", \"7\": \"layer4\"}\n        model.load_state_dict({names[k.split(\".\")[1]] + k[len(\"backbone.\") + len(k.split(\".\")[1]):]: v for k, v in sd.items()})\n    return model\n\n\nclass KneeNet(nn.Module):\n    def __init__(self, backbone: str = \"resnet34\", pretrained: bool = True, dim: int = 256, depth: int = 2,\n                 heads: int = 8, dropout: float = 0.1, drop_path: float = 0.1, pool: str = \"avg\",\n                 seq_emb: bool = False):\n        super().__init__()\n        kwargs = {\"drop_path_rate\": drop_path} if drop_path else {}\n        if backbone.startswith(\"vit_\"):  # ViTs (e.g. DINOv2) were pretrained at another size: resample positions\n            kwargs[\"dynamic_img_size\"] = True\n        self.backbone = make_backbone(backbone, pretrained, **kwargs)\n        # pool inside each window: \"avg\" (the backbone's own head) or \"avgmax\" (+ max over the feature map,\n        # through the head's norm when it has one: a small lesion is diluted by the average)\n        self.pool_type = pool\n        n_feats = self.backbone.num_features * (2 if pool == \"avgmax\" else 1)\n        self.proj = nn.Sequential(nn.Linear(n_feats, dim), nn.LayerNorm(dim))\n        self.slot_emb = nn.Parameter(torch.zeros(len(SLOTS), dim))\n        # weighting of each slot's series (T1 / PD / T2 / ...), added to its tokens; zero at the start\n        self.seq_emb = nn.Embedding(len(SEQ_CLASSES), dim) if seq_emb else None\n        if self.seq_emb is not None:\n            nn.init.zeros_(self.seq_emb.weight)\n        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))\n        layer = nn.TransformerEncoderLayer(dim, heads, dim * 2, dropout, batch_first=True, norm_first=True)\n        self.encoder = nn.TransformerEncoder(layer, depth)\n        self.queries = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)\n        self.pool = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)\n        self.norm = nn.LayerNorm(dim)\n        self.head = nn.Linear(dim, 1)\n        self.max_head = nn.Linear(dim, len(LABELS))  # max over windows: a focal lesion shows in few windows\n\n    def embed(self, imgs):\n        if self.pool_type == \"avg\":\n            return self.backbone(imgs)\n        fmap = self.backbone.forward_features(imgs)  # (N, C, h, w)\n        mx = fmap.amax((2, 3), keepdim=True)\n        norm = getattr(getattr(self.backbone, \"head\", None), \"norm\", None)  # ConvNeXt: LayerNorm2d after pooling\n        if norm is not None:\n            mx = norm(mx)\n        return torch.cat([self.backbone.forward_head(fmap, pre_logits=True), mx.flatten(1)], 1)\n\n    def forward(self, x, pos, mask, seq=None):\n        # x: (B, S, T, 3, H, W) uint8; pos: (B, S, T); mask: (B, S); seq: (B, S) SEQ_CLASSES indices\n        b, s, t = x.shape[:3]\n        imgs = x.flatten(0, 2).float()\n        imgs = (imgs - MEAN.to(imgs.device)) / STD.to(imgs.device)\n        feats = self.proj(self.embed(imgs.contiguous(memory_format=torch.channels_last))).view(b, s, t, -1)\n        tokens = feats + self.slot_emb[None, :, None] + self.pos_emb(pos.unsqueeze(-1))\n        if self.seq_emb is not None:\n            tokens = tokens + self.seq_emb(seq)[:, :, None]\n        tokens = tokens.flatten(1, 2)  # (B, S*T, D)\n        pad = ~mask.repeat_interleave(t, dim=1)  # True = ignore\n        pad[pad.all(1), 0] = False  # a study with no readable series still gets a (blank) token\n        tokens = self.encoder(tokens, src_key_padding_mask=pad)\n        q = self.queries.unsqueeze(0).expand(b, -1, -1)\n        pooled, _ = self.pool(q, tokens, tokens, key_padding_mask=pad)\n        logits = self.head(self.norm(pooled)).squeeze(-1)  # (B, 12)\n        max_logits = self.max_head(tokens).masked_fill(pad.unsqueeze(-1), -1e4).amax(1)\n        return logits + max_logits\n"}
for f, src in NEW.items():
    open(f'{KN}/{f}', 'w').write(src)
sys.path.insert(0, KN)


In [ ]:
N = 60
ts = pd.read_csv(f'{DATA}/train_series.csv')
studies = ts['StudyInstanceUID'].drop_duplicates().sample(N, random_state=0)
fake = Path('/kaggle/working/fake'); fake.mkdir(exist_ok=True)
ts[ts['StudyInstanceUID'].isin(studies)].to_csv(fake / 'test_series.csv', index=False)
pd.DataFrame({'StudyInstanceUID': studies.to_numpy()}).to_csv(fake / 'test.csv', index=False)
if not (fake / 'test_series').exists():
    os.symlink(f'{DATA}/train_series', fake / 'test_series')
from knee import infer
print('infer from', infer.__file__, 'grouped' if hasattr(infer, 'predict_group') else 'OLD')
from knee.data import StudyDataset, assign_slots
from torch.utils.data import DataLoader
W = os.cpu_count()
t = time.time()
series = infer.prepare_cache(fake, Path('/tmp/knee_cache'), W, sizes=[256, 384])
dec = time.time() - t
print(f'DECODE {N} studies, {len(series)} series: {dec:.0f}s = {dec / N:.2f} s/study (both sizes)')
slots = assign_slots(series).set_index('StudyInstanceUID').reindex(studies.to_numpy()).reset_index()
M = {os.path.basename(p).split('_')[2]: Path(p) for p in glob.glob(f'{KNEE}/models/*')}
print(sorted(M))

In [ ]:
import json
V27 = ['e46', 'e53', 'e39', 'e42', 'e50', 'e36', 'e47', 'e38', 'e41', 'e43', 'e44', 'e35', 'e45', 'e37', 'e32', 'e34', 'e30', 'e33']
mds = [M[k] for k in V27]
cfgs = {m: json.loads((m / 'config.json').read_text()) for m in mds}
cache = Path('/tmp/knee_cache')
t = time.time()
P = infer._run_grouped(mds, cfgs, slots, cache, W, 24, None, time.time(), 16)
g = time.time() - t
print(f'GROUPED 18 models, 2 GPUs, 24 windows: {g:.0f}s = {g / N:.2f} s/study', flush=True)
for i in range(torch.cuda.device_count()):
    print(f'cuda:{i} peak memory {torch.cuda.max_memory_allocated(i) / 2**30:.1f} GiB', flush=True)
for k in ['e46', 'e41', 'e37']:
    ref = infer.predict_model(M[k], slots, infer.size_dir(cache, cfgs[M[k]].get('cache_size', 256)), torch.device('cuda:0'), W, 24)
    print(f'CHECK {k}: max |grouped - per-model| = {float(np.abs(P[M[k]] - ref).max()):.2e}', flush=True)

In [ ]:
# budget logic: 18 models with a 5-minute budget on the same 60 studies (calibration on 16)
t = time.time()
P2 = infer._run_grouped(mds, cfgs, slots, cache, W, 24, 300, time.time(), 16)
print(f'BUDGET 300 s: {time.time() - t:.0f}s, {len(P2)} of 18 models kept: {[m.name.split("_")[2] for m in P2]}', flush=True)
print(f'total {(time.time() - T0) / 60:.1f} min')